# Projet 12 - Detection de faux billets
## Demonstration : chargement du modele et inference sur de nouvelles donnees

Ce notebook accompagne `p12_faux_billets.ipynb` (notebook d'analyse principal).
Il montre comment charger le modele entraine (`model_faux_billets.joblib`) et
l'utiliser pour predire de nouveaux billets, de deux facons :

1. directement dans ce notebook, avec `joblib.load` et `pipeline.predict` ;
2. via le script de production `predict_billets.py`, livre a part.

Prerequis : `model_faux_billets.joblib` et `predict_billets.py` doivent etre
dans le meme dossier que ce notebook.

# 1) Chargement du modele et inference directe

On charge le pipeline complet (imputation KNNImputer + standardisation +
regression logistique) sauvegarde a la fin du notebook d'analyse. Comme
KNNImputer et StandardScaler sont des classes standard de scikit-learn, le
chargement ne necessite aucun module externe.

In [1]:
import joblib
import pandas as pd

pipeline = joblib.load("model_faux_billets.joblib")
print("Modele charge :", pipeline)

Modele charge : Pipeline(steps=[('knnimputer', KNNImputer()),
                ('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(max_iter=1000, random_state=42))])


## 1.1) Inference sur un fichier CSV

In [2]:
billets = pd.read_csv("billets.csv", sep=";")  # remplacer par billets_production.csv en conditions reelles
colonnes = ["diagonal", "height_left", "height_right", "margin_low", "margin_up", "length"]

predictions = pipeline.predict(billets[colonnes])
probabilites = pipeline.predict_proba(billets[colonnes])

resultat = billets.copy()
resultat["prediction"] = ["Vrai" if p else "Faux" for p in predictions]
resultat["probabilite_faux"] = probabilites[:, 0].round(4)
resultat["probabilite_vrai"] = probabilites[:, 1].round(4)

resultat.head(10)

,is_genuine,diagonal,height_left,height_right,margin_low,margin_up,length,prediction,probabilite_faux,probabilite_vrai
0,True,171.81,104.86,104.95,4.52,2.89,112.83,Vrai,0.2770,0.7230
1,True,171.46,103.36,103.66,3.77,2.99,113.09,Vrai,0.0001,0.9999
2,True,172.69,104.48,103.50,4.40,2.94,113.16,Vrai,0.0016,0.9984
3,True,171.36,103.91,103.94,3.62,3.01,113.51,Vrai,0.0001,0.9999
4,True,171.73,104.28,103.46,4.04,3.48,112.54,Vrai,0.2555,0.7445
5,True,172.17,103.74,104.08,4.42,2.95,112.81,Vrai,0.0105,0.9895
6,True,172.34,104.18,103.85,4.58,3.26,112.81,Vrai,0.1895,0.8105
7,True,171.88,103.76,104.08,3.98,2.92,113.08,Vrai,0.0006,0.9994
8,True,172.47,103.92,103.67,4.00,3.25,112.85,Vrai,0.0081,0.9919
9,True,172.47,104.07,104.02,4.04,3.25,113.45,Vrai,0.0020,0.9980


## 1.2) Inference sur un billet unique

In [3]:
un_billet = pd.DataFrame({
    "diagonal": [171.8],
    "height_left": [104.0],
    "height_right": [104.2],
    "margin_low": [4.3],
    "margin_up": [3.1],
    "length": [113.1],
})

prediction = pipeline.predict(un_billet)[0]
proba = pipeline.predict_proba(un_billet)[0]

print("Prediction :", "VRAI" if prediction else "FAUX")
print(f"Probabilite vrai billet : {proba[1]*100:.2f}%")
print(f"Probabilite faux billet : {proba[0]*100:.2f}%")

Prediction : VRAI
Probabilite vrai billet : 98.59%
Probabilite faux billet : 1.41%


## 1.3) Inference avec une valeur manquante (imputation automatique)

In [4]:
billet_incomplet = pd.DataFrame({
    "diagonal": [171.9],
    "height_left": [104.3],
    "height_right": [104.5],
    "margin_low": [None],
    "margin_up": [3.4],
    "length": [111.5],
})

prediction = pipeline.predict(billet_incomplet)[0]
proba = pipeline.predict_proba(billet_incomplet)[0]

print("Prediction :", "VRAI" if prediction else "FAUX")
print(f"Probabilite vrai billet : {proba[1]*100:.2f}%")
print(f"Probabilite faux billet : {proba[0]*100:.2f}%")

Prediction : FAUX
Probabilite vrai billet : 0.02%
Probabilite faux billet : 99.98%


La valeur manquante de `margin_low` est imputee automatiquement par le
KNNImputer integre au pipeline, sans aucune action supplementaire de
notre part.

# 2) Demonstration via le script de production

En plus de l'utilisation directe ci-dessus, le pipeline est encapsule dans
un script autonome, `predict_billets.py`, destine a etre utilise en ligne
de commande par les equipes de l'ONCFM (sans avoir besoin d'ouvrir un
notebook).

## 2.1) Contenu du script

In [5]:
with open("predict_billets.py", encoding="utf-8") as f:
    print(f.read())

"""
Script de production - Detection de faux billets (ONCFM)

Ce script charge le modele entraine (pipeline complet : imputation +
standardisation + Random Forest) et permet de predire si un ou plusieurs
billets sont vrais ou faux, a partir de leurs caracteristiques geometriques.

PREREQUIS (2 fichiers doivent etre dans le meme dossier) :
    - predict_billets.py       (ce script)
    - model_faux_billets.joblib (le modele entraine)

UTILISATION
-----------

1) Predire un fichier CSV contenant plusieurs billets :
    python predict_billets.py --csv billets_production.csv

   Le fichier CSV doit contenir les colonnes :
   diagonal, height_left, height_right, margin_low, margin_up, length
   (separateur ";", comme le fichier d'exemple billets.csv)

   Un fichier de resultats est cree a cote (par defaut : resultats_predictions.csv),
   et un resume est affiche dans le terminal.

2) Predire un seul billet en donnant ses valeurs directement :
    python predict_billets.py --diagonal 171.8 -

## 2.2) Demonstration - prediction sur un fichier CSV

On utilise `billets.csv` comme exemple (le script ignore la colonne
`is_genuine` en trop, il ne se sert que des 6 colonnes geometriques).

In [6]:
!python predict_billets.py --csv billets.csv --output demo_resultats.csv


1500 billets analyses :
  - 1006 vrais billets
  - 494 faux billets

Resultats detailles ecrits dans : demo_resultats.csv


In [7]:
demo = pd.read_csv("demo_resultats.csv", sep=";")
demo.head(10)

,is_genuine,diagonal,height_left,height_right,margin_low,margin_up,length,prediction,probabilite_faux,probabilite_vrai
0,True,171.81,104.86,104.95,4.52,2.89,112.83,Vrai,0.2770,0.7230
1,True,171.46,103.36,103.66,3.77,2.99,113.09,Vrai,0.0001,0.9999
2,True,172.69,104.48,103.50,4.40,2.94,113.16,Vrai,0.0016,0.9984
3,True,171.36,103.91,103.94,3.62,3.01,113.51,Vrai,0.0001,0.9999
4,True,171.73,104.28,103.46,4.04,3.48,112.54,Vrai,0.2555,0.7445
5,True,172.17,103.74,104.08,4.42,2.95,112.81,Vrai,0.0105,0.9895
6,True,172.34,104.18,103.85,4.58,3.26,112.81,Vrai,0.1895,0.8105
7,True,171.88,103.76,104.08,3.98,2.92,113.08,Vrai,0.0006,0.9994
8,True,172.47,103.92,103.67,4.00,3.25,112.85,Vrai,0.0081,0.9919
9,True,172.47,104.07,104.02,4.04,3.25,113.45,Vrai,0.0020,0.9980


## 2.3) Demonstration - prediction d'un billet unique

In [8]:
!python predict_billets.py --diagonal 171.8 --height_left 104.0 --height_right 104.2 --margin_low 4.3 --margin_up 3.1 --length 113.1


Prediction : VRAI
Probabilite vrai billet : 98.59%
Probabilite faux billet : 1.41%


## 2.4) Demonstration - billet sans margin_low (imputation automatique)

In [9]:
!python predict_billets.py --diagonal 171.9 --height_left 104.3 --height_right 104.5 --margin_up 3.4 --length 111.5


Prediction : FAUX
Probabilite vrai billet : 0.02%
Probabilite faux billet : 99.98%


Les 3 demonstrations confirment que le script fonctionne de bout en bout :
prediction sur un fichier de plusieurs billets (avec fichier de resultats
genere), prediction d'un billet unique saisi en ligne de commande, et
gestion automatique d'une valeur manquante (`margin_low`) grace au
KNNImputer integre au pipeline sauvegarde.